In [ ]:
import networkx as nx
from rdflib import Graph, Namespace, BNode, URIRef, Literal
from pyvis import network as net

In [ ]:
g = Graph()
g.parse("graph.jelly", format="jelly")
ns = Namespace("https://schema.org")

In [ ]:
def short_label(node) -> str:
    """
    Return a concise, human-readable label for an RDF node.

    Strips the namespace prefix from URIs, keeping only the fragment
    (``#name``) or last path segment (``/name``).

    Parameters
    ----------
    node : rdflib.term.Node
        An RDF node — one of ``URIRef``, ``Literal``, or ``BNode``.

    Returns
    -------
    str
        Shortened string representation suitable for use as a node label.
    """
    if isinstance(node, URIRef):
        uri = str(node)
        if "#" in uri:
            return uri.split("#")[-1]
        elif "schema.org" in uri:
            return uri.split("schema.org")[-1]
        return uri.rstrip("/").split("/")[-1] or uri
    if isinstance(node, Literal):
        return str(node)
    if isinstance(node, BNode):
        return f"_:{str(node)}"
    return str(node)

In [ ]:
G = nx.DiGraph()
for s, p, o in g:
    s, p, o = short_label(s), short_label(p), short_label(o)
    if ".jsonld" in str(o) or ".jsonld" in str(p):
        if "B5D" in str(o) or "b5d" in str(p):
            continue
        G.add_edge(s, o, label=p)
        print(s, p, o)
    else:
        continue

pyg = net.Network(
    notebook=False,           # export for browser, not notebook cell
    cdn_resources="remote",   # or "in_line"
    width="90%",              # NOT 100% width or 100% height
    height="800px"            # give it finite space
)
pyg.from_nx(G)

# activate the panel, then show in browser
pyg.show_buttons()
pyg.show("Test.html", notebook=False)   # opens in browser

In [ ]:
G = nx.DiGraph()
i = 0
for s, p, o in g:
    if i < 5:
        print("s: ", s)
        print("s: ", short_label(s))
        print("p: ", p)
        print("p: ", short_label(p))
        print("o: ", o)
        print("o: ", short_label(o))
    i += 1
    G.add_edge(short_label(s), short_label(o), label=short_label(p))

In [ ]:
for triple in sorted(G):
    print(triple)

In [ ]:
#nx.write_gexf(G, "DatahubTest.gexf")

In [ ]:
#pyg = net.Network(notebook=True, cdn_resources='in_line', height="600px", width="100%")
#pyg.from_nx(G)
#pyg.show("DatahubTest.html")

In [ ]:
# Find all entities that have @id properties
entity_ids = {}
all_predicates = set()
for s, p, o in g:
    all_predicates.add(str(p))
    # Look for various ways @id might be represented
    if str(p) == "@id" or str(p).endswith("#id") or str(p).endswith("/id") or "id" in str(p).lower():
        entity_ids[s] = str(o)

print(f"All unique predicates found: {sorted(all_predicates)}")
print(f"Found {len(entity_ids)} entities with @id properties")
for entity, entity_id in list(entity_ids.items())[:5]:
    print(f"Entity: {short_label(entity)} -> ID: {entity_id}")

# If no @id found, let's try a different approach - look for any property that contains "id"
if not entity_ids:
    print("No @id properties found, trying alternative approach...")
    for s, p, o in g:
        if "id" in str(p).lower() and isinstance(o, (URIRef, Literal)):
            entity_ids[s] = str(o)
            if len(entity_ids) >= 10:  # Limit to first 10
                break
    print(f"Alternative approach found {len(entity_ids)} entities")
    for entity, entity_id in list(entity_ids.items())[:5]:
        print(f"Entity: {short_label(entity)} -> ID: {entity_id}")

In [ ]:
# Create filtered graph with only @id entities as nodes
G_filtered = nx.DiGraph()

# Add nodes for entities with @id
for entity, entity_id in entity_ids.items():
    G_filtered.add_node(entity_id, label=short_label(entity))

# Add edges between entities that have relationships
for s, p, o in g:
    # Only add edges if both subject and object are entities with @id
    if s in entity_ids and o in entity_ids:
        source_id = entity_ids[s]
        target_id = entity_ids[o]
        G_filtered.add_edge(source_id, target_id, label=short_label(p))

print(f"Filtered graph has {len(G_filtered.nodes)} nodes and {len(G_filtered.edges)} edges")

In [ ]:
# Print some sample edges from filtered graph
for i, (source, target, data) in enumerate(G_filtered.edges(data=True)):
    if i < 10:
        print(f"{source} --{data['label']}--> {target}")

In [ ]:
nx.write_gexf(G_filtered, "DatahubTest_filtered.gexf")

In [ ]:
pyg_filtered = net.Network(notebook=True, cdn_resources='in_line', height="600px", width="100%")
pyg_filtered.from_nx(G_filtered)
pyg_filtered.show("DatahubTest_filtered.html")